In [9]:
import pandas as pd
import numpy as np

df = pd.read_csv("census_population_2022_geocoded_google_refined.csv", dtype=str)

def decimal_places(x):
    if pd.isna(x):
        return -1
    x = str(x).strip()
    if "." not in x:
        return 0
    return len(x.split(".")[-1].rstrip("0"))

df["lat_decimals"] = df["latitude"].apply(decimal_places)
df["lon_decimals"] = df["longitude"].apply(decimal_places)

# flag coarse coordinates
df["needs_reassign"] = (df["lat_decimals"] <= 3) | (df["lon_decimals"] <= 3)

print(df["needs_reassign"].value_counts())

summary = (
    df.groupby("Census District")["needs_reassign"]
    .agg(
        total="count",
        needs_reassign="sum"
    )
)

summary["pct_reassign"] = summary["needs_reassign"] / summary["total"]

# sort to see worst districts first
summary = summary.sort_values("pct_reassign", ascending=False)

print(summary)

needs_reassign
True     7079
False    1183
Name: count, dtype: int64
                                total  needs_reassign  pct_reassign
Census District                                                    
Central Kgalagadi Game Reserve     12              12      1.000000
Central Boteti                    593             561      0.946037
Ghanzi                            582             546      0.938144
Central Bobonong                  464             431      0.928879
South East                        137             127      0.927007
Kweneng West                      601             552      0.918469
Central Serowe/ Palapye          1011             924      0.913947
Ngamiland East                    437             395      0.903890
Kgatleng                          350             313      0.894286
Ngwaketse                         459             410      0.893246
Central Mahalapye                 598             523      0.874582
Ngwaketse West                    260          

In [10]:
# --------------------------------------------------
# 1. load current edited file
# --------------------------------------------------
df = pd.read_csv("census_population_2022_geocoded_google_refined.csv")

# --------------------------------------------------
# 2. identify key columns
# --------------------------------------------------
district_col = None
pop_col = None

for col in df.columns:
    col_lower = col.lower()
    if "district" in col_lower:
        district_col = col
    if "pop" in col_lower:
        pop_col = col

if district_col is None:
    raise KeyError(f"Could not find district column. Available columns: {list(df.columns)}")

if pop_col is None:
    raise KeyError(f"Could not find population column. Available columns: {list(df.columns)}")

print(f"Using district column: {district_col}")
print(f"Using population column: {pop_col}")

# --------------------------------------------------
# 3. make lat/lon numeric
# --------------------------------------------------
df["latitude"] = pd.to_numeric(df["latitude"], errors="coerce")
df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")

# --------------------------------------------------
# 4. helper to count decimal places
# --------------------------------------------------
def decimal_places(x):
    if pd.isna(x):
        return -1
    s = str(x).strip()
    if "." not in s:
        return 0
    return len(s.split(".")[-1].rstrip("0"))

df["lat_decimals"] = df["latitude"].apply(decimal_places)
df["lon_decimals"] = df["longitude"].apply(decimal_places)

# --------------------------------------------------
# 5. flag low-precision rows
#    reassign if either latitude or longitude has < 3 decimals
# --------------------------------------------------
df["needs_reassign"] = (
    df["latitude"].isna() |
    df["longitude"].isna() |
    (df["lat_decimals"] < 3) |
    (df["lon_decimals"] < 3)
)

df["good_coords"] = ~df["needs_reassign"]

# --------------------------------------------------
# 6. clean population column
# --------------------------------------------------
# handles strings like "48,431"
df["pop_clean"] = (
    df[pop_col]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.strip()
)

# convert empty-ish strings to NaN, then numeric
df["pop_clean"] = pd.to_numeric(df["pop_clean"], errors="coerce")

if df["pop_clean"].isna().any():
    print(f"Warning: {df['pop_clean'].isna().sum()} rows have non-numeric population values.")

# --------------------------------------------------
# 7. overall row coverage
# --------------------------------------------------
total_rows = len(df)
good_rows = int(df["good_coords"].sum())
bad_rows = int(df["needs_reassign"].sum())

print("\nROW COVERAGE")
print(f"total rows: {total_rows:,}")
print(f"good rows: {good_rows:,}")
print(f"rows needing reassign: {bad_rows:,}")
print(f"percent rows accounted for: {good_rows / total_rows:.2%}")

# --------------------------------------------------
# 8. overall population coverage
# --------------------------------------------------
total_pop = df["pop_clean"].sum()
good_pop = df.loc[df["good_coords"], "pop_clean"].sum()
bad_pop = df.loc[df["needs_reassign"], "pop_clean"].sum()

print("\nPOPULATION COVERAGE")
print(f"total population: {total_pop:,.0f}")
print(f"population with good coords: {good_pop:,.0f}")
print(f"population needing reassign: {bad_pop:,.0f}")
print(f"percent population accounted for: {good_pop / total_pop:.2%}")

# --------------------------------------------------
# 9. district-by-district row summary
# --------------------------------------------------
district_row_summary = (
    df.groupby(district_col)["needs_reassign"]
    .agg(total_rows="count", rows_needing_reassign="sum")
)

district_row_summary["good_rows"] = (
    district_row_summary["total_rows"] - district_row_summary["rows_needing_reassign"]
)
district_row_summary["pct_rows_accounted_for"] = (
    district_row_summary["good_rows"] / district_row_summary["total_rows"]
)

district_row_summary = district_row_summary.sort_values(
    "pct_rows_accounted_for", ascending=True
)

print("\nDISTRICT ROW COVERAGE")
print(district_row_summary)

# --------------------------------------------------
# 10. district-by-district population summary
# --------------------------------------------------
district_pop_summary = (
    df.groupby(district_col)
    .apply(
        lambda x: pd.Series({
            "total_population": x["pop_clean"].sum(),
            "population_needing_reassign": x.loc[x["needs_reassign"], "pop_clean"].sum(),
            "population_accounted_for": x.loc[x["good_coords"], "pop_clean"].sum(),
        })
    )
)

district_pop_summary["pct_population_accounted_for"] = (
    district_pop_summary["population_accounted_for"] / district_pop_summary["total_population"]
)

district_pop_summary = district_pop_summary.sort_values(
    "pct_population_accounted_for", ascending=True
)

print("\nDISTRICT POPULATION COVERAGE")
print(district_pop_summary)

# --------------------------------------------------
# 11. optional: save summaries
# --------------------------------------------------
district_row_summary.to_csv("district_row_coverage_summary.csv")
district_pop_summary.to_csv("district_population_coverage_summary.csv")

print("\nSaved:")
print("- district_row_coverage_summary.csv")
print("- district_population_coverage_summary.csv")

Using district column: Census District
Using population column: Total Population

ROW COVERAGE
total rows: 8,262
good rows: 1,183
rows needing reassign: 7,079
percent rows accounted for: 14.32%

POPULATION COVERAGE
total population: 4,273,386
population with good coords: 3,790,916
population needing reassign: 482,470
percent population accounted for: 88.71%

DISTRICT ROW COVERAGE
                                total_rows  rows_needing_reassign  good_rows  \
Census District                                                                
Central Kgalagadi Game Reserve          12                     12          0   
Central Boteti                         593                    561         32   
Ghanzi                                 582                    546         36   
Central Bobonong                       464                    431         33   
South East                             137                    127         10   
Kweneng West                           601               

In [11]:
name_col = "City/Town/Village"

mask_assoc = df[name_col].str.lower().str.contains("associated localities", na=False)

print(f"Dropping {mask_assoc.sum()} rows with 'associated localities'")

df = df[~mask_assoc].copy()

Dropping 498 rows with 'associated localities'


In [12]:
total_pop = df["pop_clean"].sum()
print(f"New total population: {total_pop:,.0f}")

New total population: 2,392,866


In [ ]:
good_pop = df.loc[df["good_coords"], "pop_clean"].sum()

print(f"Corrected total population: {total_pop:,.0f}")
print(f"Corrected coverage: {good_pop / total_pop:.2%}")

Corrected total population: 2,392,866
Corrected coverage: 82.86%


In [14]:
original_cols = df.columns.tolist()

lat_num = pd.to_numeric(df["latitude"], errors="coerce")
lon_num = pd.to_numeric(df["longitude"], errors="coerce")

def decimal_places(x):
    if pd.isna(x):
        return -1
    s = str(x).strip()
    if "." not in s:
        return 0
    return len(s.split(".")[-1].rstrip("0"))

lat_dec = lat_num.apply(decimal_places)
lon_dec = lon_num.apply(decimal_places)

needs_reassign = (
    lat_num.isna() |
    lon_num.isna() |
    (lat_dec < 3) |
    (lon_dec < 3)
)

district_bounds = {
    "Barolong": {"lat_min": -25.641933, "lat_max": -25.265160, "lon_min": 24.242183, "lon_max": 25.580426},
    "Central Bobonong": {"lat_min": -22.557583, "lat_max": -21.513765, "lon_min": 27.497297, "lon_max": 29.293562},
    "Central Boteti": {"lat_min": -21.760687, "lat_max": -20.294313, "lon_min": 23.887281, "lon_max": 25.590161},
    "Central Kalahari Game Reserve": {"lat_min": -23.333333, "lat_max": -21.000000, "lon_min": 22.750000, "lon_max": 25.333333},
    "Central Kgalagadi Game Reserve": {"lat_min": -26.334848, "lat_max": -24.108726, "lon_min": 20.019343, "lon_max": 22.106745},
    "Central Mahalapye": {"lat_min": -23.441585, "lat_max": -21.843818, "lon_min": 24.798611, "lon_max": 27.390120},
    "Central Serowe/ Palapye": {"lat_min": -22.932435, "lat_max": -21.275532, "lon_min": 26.166656, "lon_max": 28.024451},
    "Central Tutume": {"lat_min": -21.833471, "lat_max": -19.064729, "lon_min": 24.400654, "lon_max": 27.432881},
    "Chobe": {"lat_min": -18.979529, "lat_max": -17.789564, "lon_min": 23.886376, "lon_max": 25.939425},
    "Francistown": {"lat_min": -21.219025, "lat_max": -21.108503, "lon_min": 27.449010, "lon_max": 27.571509},
    "Gaborone": {"lat_min": -24.692249, "lat_max": -24.577262, "lon_min": 25.796392, "lon_max": 25.997278},
    "Ghanzi": {"lat_min": -23.295084, "lat_max": -21.065592, "lon_min": 19.981551, "lon_max": 25.524456},
    "Jwaneng": {"lat_min": -25.262560, "lat_max": -24.009316, "lon_min": 23.057140, "lon_max": 25.084118},
    "Kgalagadi North": {"lat_min": -24.768852, "lat_max": -23.354433, "lon_min": 20.030329, "lon_max": 23.106501},
    "Kgalagadi South": {"lat_min": -26.709681, "lat_max": -24.150146, "lon_min": 20.031313, "lon_max": 24.436830},
    "Kgatleng": {"lat_min": -24.660912, "lat_max": -23.580518, "lon_min": 25.990498, "lon_max": 26.918842},
    "Kweneng East": {"lat_min": -24.734424, "lat_max": -22.955670, "lon_min": 25.140448, "lon_max": 25.932800},
    "Kweneng West": {"lat_min": -24.542678, "lat_max": -23.332725, "lon_min": 22.901756, "lon_max": 25.247337},
    "Lobatse": {"lat_min": -25.249047, "lat_max": -25.155074, "lon_min": 25.654378, "lon_max": 25.720041},
    "Ngamiland Delta": {"lat_min": -20.150573, "lat_max": -18.341372, "lon_min": 21.031319, "lon_max": 23.879192},
    "Ngamiland West": {"lat_min": -21.022556, "lat_max": -19.777409, "lon_min": 21.023684, "lon_max": 23.894462},
    "Ngamiland East": {"lat_min": -20.338843, "lat_max": -19.009089, "lon_min": 23.893024, "lon_max": 25.195922},
    "Ngwaketse": {"lat_min": -25.269546, "lat_max": -24.811678, "lon_min": 24.859198, "lon_max": 25.603521},
    "Ngwaketse West": {"lat_min": -24.965602, "lat_max": -24.588284, "lon_min": 25.057838, "lon_max": 25.686650},
    "North East": {"lat_min": -21.602388, "lat_max": -20.533807, "lon_min": 27.212563, "lon_max": 28.046629},
    "Orapa": {"lat_min": -22.301559, "lat_max": -20.994533, "lon_min": 24.287474, "lon_max": 26.221068},
    "Selibe Phikwe": {"lat_min": -22.066612, "lat_max": -21.936743, "lon_min": 27.793017, "lon_max": 27.874316},
    "South East": {"lat_min": -25.465129, "lat_max": -24.481681, "lon_min": 25.649737, "lon_max": 26.069964},
    "Sowa": {"lat_min": -20.671401, "lat_max": -20.498575, "lon_min": 26.116719, "lon_max": 26.310985},
}

rng = np.random.default_rng(42)

for district, bounds in district_bounds.items():
    idx = df.index[
        (df["Census District"] == district) &
        (needs_reassign)
    ]
    n = len(idx)
    if n == 0:
        continue
    df.loc[idx, "latitude"] = rng.uniform(bounds["lat_min"], bounds["lat_max"], size=n)
    df.loc[idx, "longitude"] = rng.uniform(bounds["lon_min"], bounds["lon_max"], size=n)

df = df[original_cols]

df.to_csv("census_population_2022_geocoded_final_uniform.csv", index=False)